# Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

/home/seongyoonjeon/venvs/lg-aimers-hackathon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setting

In [2]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 2048
MAX_SEQUENCE_LENGTH = 2048

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

DAMPENING_FRAC = 0.02
BLOCK_SIZE = 256 # 128 instead 256이면 성능 낮음, 속도 빠름

In [3]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)

torch version: 2.9.1+cu130
cuda available: True
torch cuda version: 13.0


In [4]:
# GPU 메모리 상황 모니터링
from pynvml import *

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)
info = nvmlDeviceGetMemoryInfo(handle)

print(f"Total: {info.total / 1024**2:.1f} MB")
print(f"Used : {info.used / 1024**2:.1f} MB")
print(f"Free : {info.free / 1024**2:.1f} MB")

Total: 12288.0 MB
Used : 962.7 MB
Free : 11325.3 MB


# Model Loads

In [5]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",

    low_cpu_mem_usage=True,  # 추가
    max_memory={0: "10GiB", "cpu": "20GiB"},  # GPU 메모리 여유 확보
)

print("[INFO] 모델/토크나이저 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [6]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


Map: 100%|██████████| 2048/2048 [00:00<00:00, 7778.91 examples/s]

[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [7]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

# 양자화 전 메모리 정리
import gc
torch.cuda.empty_cache()
gc.collect()

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        
        dampening_frac=DAMPENING_FRAC,
        block_size=BLOCK_SIZE,
    )
]

# GPTQ 시작 전에 추가
def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"[MEM] Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

print_gpu_memory()

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,

    batch_size=1,  # 배치 크기 최소화
    
    # 데이터 처리 최적화
    text_column="text",
    pad_to_max_length=False,  # 패딩 비활성화로 메모리 절약
    shuffle_calibration_samples=True,
    
    # 캐시 및 전처리
    overwrite_cache=True,
    preprocessing_num_workers=1,  # 워커 수 제한
    
    # 양자화 설정
    quantization_aware_calibration=True,
)

print_gpu_memory()

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=2048, max_len=2048)...
[MEM] Allocated: 2.38GB, Reserved: 2.39GB


Tokenizing (num_proc=1): 100%|██████████| 2048/2048 [00:02<00:00, 699.07 examples/s]

2026-02-10T17:21:07.165961+0900 | reset | INFO - Compression lifecycle reset
2026-02-10T17:21:07.167666+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-10T17:21:07.207961+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-10T17:21:07.208483+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(1/31): Calibrating: 100%|██████████| 2048/2048 [00:15<00:00, 129.34it/s]

2026-02-10T17:21:24.979090+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 2048 samples


...

(30/31): Calibrating: 100%|██████████| 2048/2048 [00:15<00:00, 130.75it/s]

2026-02-10T17:33:29.960814+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 2048 samples


2026-02-10T17:33:30.305195+0900 | compress | METRIC - time 0.34s
2026-02-10T17:33:30.306019+0900 | compress | METRIC - error 5775.93
2026-02-10T17:33:30.306346+0900 | compress | METRIC - GPU 0 | usage: 17.32% | total memory: 12 GB
2026-02-10T17:33:30.306644+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-10T17:33:30.307128+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 2048 samples
2026-02-10T17:33:30.629849+0900 | compress | METRIC - time 0.32s
2026-02-10T17:33:30.630738+0900 | compress | METRIC - error 1642.07
2026-02-10T17:33:30.631108+0900 | compress | METRIC - GPU 0 | usage: 17.32% | total memory: 12 GB
2026-02-10T17:33:30.631312+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-10T17:33:30.631613+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 2048 samples
2026-02-10T17:33:30.954075+0900 | compress | METRIC - time 0.32s
2026-02-10T17:33:30.954946+0900 | compress | ME

(31/31): Propagating: 100%|██████████| 2048/2048 [00:01<00:00, 1667.30it/s]

2026-02-10T17:33:40.989573+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-10T17:33:41.014154+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[MEM] Allocated: 0.01GB, Reserved: 0.41GB
[INFO] GPTQ 완료


# Model Save

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-10T17:33:41.031015+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:02, 78.52it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [9]:
zip_name = "submit-ver9"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] submit-ver9.zip 생성 중...
[INFO] 생성 완료: submit-ver9.zip
